In [1]:
import sys


sys.path.append('../')

from bunkatopics import Bunka
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset
import random

# import umap
from umap.umap_ import UMAP # My personal Umap bugs so I use this one
from sentence_transformers import SentenceTransformer
random.seed(42)


model_name = "all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=model_name) # We recommend starting with a small model

In [2]:

#Scientific Litterature Data
dataset = load_dataset("CShorten/ML-ArXiv-Papers")["train"]["title"]
raw_docs = random.sample(dataset, 1000)


projection_model = UMAP(
                n_components=2,
                random_state=42,
                n_neighbors=5,# I want to optimise the local structure (5 low, 25 high)
                min_dist = 0.3,
                metric = 'euclidean') # I don't want to disperse embeddings

# #embedding_model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer(model_name_or_path="Bunka/sentence_transformer_encoder")

projection_model.n_components

2

In [3]:
bunka = Bunka(embedding_model=embedding_model, 
                projection_model=projection_model)  # the language is automatically detected, make sure the embedding model is adapted

In [4]:
bunka.actual_dimensions

2

In [5]:
# Fit Bunka to your text data
bunka.fit(raw_docs, sampling_size_for_terms=1000)

2025-05-09 08:47:37 - Bunka - INFO - Processing 14501 tokens
2025-05-09 08:47:38 - Bunka - INFO - Detected language: English
2025-05-09 08:47:38 - Bunka - INFO - Embedding documents... (can take varying amounts of time depending on their size)
2025-05-09 08:47:40 - Bunka - INFO - Reducing dimensions to 2 using UMAP
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-05-09 08:47:45 - Bunka - INFO - Extracting meaningful terms from documents...
2025-05-09 08:47:45 - Bunka - INFO - Sampling 1000 documents for term extraction
100%|██████████| 1000/1000 [00:05<00:00, 179.90it/s]


In [6]:
bunka.fig_embeddings

In [7]:
len(bunka.docs[0].embedding)
len(bunka.docs[0].nd_embedding)

0

In [8]:
from sklearn.cluster import HDBSCAN

max_cluster_size = int(0.02*len(raw_docs))
max_cluster_size = 30
min_cluster_size = max(2, int(0.003*len(raw_docs)))
min_cluster_size = 7

clustering_model = HDBSCAN(min_samples = 1, 
                max_cluster_size=max_cluster_size, 
                min_cluster_size=min_cluster_size, 
                metric = 'euclidean',
                cluster_selection_method = 'leaf')


df_topics = bunka.get_topics(n_clusters=10, 
                             name_length=5,
                             min_count_terms = 2,  
                             custom_clustering_model=clustering_model, 
                             umap_target_weight=0) # Specify the number of terms to describe each topic

2025-05-09 08:50:16 - Bunka - INFO - Computing the topics


In [9]:
bunka.visualize_topics()

2025-05-09 08:50:18 - Bunka - INFO - Creating the Bunka Map


In [16]:
bunka.visualize_topics()

2025-05-08 20:51:12 - Bunka - INFO - Creating the Bunka Map


In [9]:
# import numpy as  np

# doc_nd_embeddings = []
# doc_indices = []
# for i, doc in enumerate(bunka.docs):
#     if hasattr(doc, "nd_embedding") and doc.nd_embedding:
#         doc_nd_embeddings.append(doc.nd_embedding)
#         doc_indices.append(i)

#         # Extract topic_ids from documents as targets for supervision
#         doc_topic_ids = []
#         for doc_idx in doc_indices:
#             topic_id = bunka.docs[doc_idx].topic_id
#             # Handle None values - map to a default value or filter out
#             if topic_id is None:
#                 topic_id = (
#                     -1
#                 )  # Use -1 or another value to indicate "no topic"
#             doc_topic_ids.append(topic_id)

#         # Convert to numpy array
#         doc_topic_ids = np.array(doc_topic_ids)

2025-05-08 20:50:16 - Bunka - INFO - Creating the Bunka Map


In [11]:
## Add it if it is 2 emebnddings

bunka.docs[0].nd_embedding

[7.425808906555176,
 8.124151229858398,
 4.166311264038086,
 5.1330180168151855,
 2.830909490585327]